In [1]:
import os, sys, json, time, random, warnings, gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import soundfile as sf

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

print("Python:", sys.executable)
print("librosa:", librosa.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable. Fix PyTorch/CUDA before training.")

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2), "GB")


Python: /home/adminbum/ntqai/.venv/bin/python
librosa: 1.0.0
PyTorch: 2.14.1+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Ti SUPER
CUDA: 13.0
VRAM: 15.99 GB


/home/adminbum/ntqai/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_ROOT = Path("/mnt/d/Data/VNTM3")
STFT_ROOT = DATA_ROOT / "_stft30_v1"
STFT_MANIFEST = STFT_ROOT / "reports" / "stft_feature_manifest.csv"

RUN_ROOT = DATA_ROOT / "_cnn2d_mel30_multiseed_v1"
MEL_DIR = RUN_ROOT / "mel_features"
CACHE_DIR = RUN_ROOT / "cache_256x320"
RESULT_ROOT = RUN_ROOT / "results"
CHECKPOINT_ROOT = RUN_ROOT / "checkpoints"
FIGURE_ROOT = RUN_ROOT / "figures"

for p in [RUN_ROOT, MEL_DIR, CACHE_DIR, RESULT_ROOT, CHECKPOINT_ROOT, FIGURE_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 123, 2026]

SR = 22050
DURATION = 30.0
TARGET_SAMPLES = int(SR * DURATION)

N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
FMIN = 0
FMAX = SR / 2
TOP_DB = 80

INPUT_H = 256
INPUT_W = 320

BATCH_SIZE = 96
NUM_WORKERS = 8
PREFETCH_FACTOR = 2

EPOCHS = 50
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 10
MIN_DELTA = 1e-4

DEVICE = torch.device("cuda")

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

print("Output:", RUN_ROOT)


Output: /mnt/d/Data/VNTM3/_cnn2d_mel30_multiseed_v1


In [3]:
base_df = pd.read_csv(STFT_MANIFEST)

required = {"split", "label", "filename"}
missing = required - set(base_df.columns)
if missing:
    raise ValueError(f"Missing columns in STFT manifest: {missing}")

classes = sorted(base_df["label"].unique())
class_to_idx = {c:i for i,c in enumerate(classes)}
idx_to_class = {i:c for c,i in class_to_idx.items()}
NUM_CLASSES = len(classes)

def resolve_audio_path(row):
    
    for col in ["audio_path", "source_path", "wav_path", "path"]:
        if col in base_df.columns and pd.notna(row.get(col)):
            p = Path(str(row[col]))
            if p.exists():
                return p

    
    p = DATA_ROOT / str(row["label"]) / str(row["filename"])
    if p.exists():
        return p

    
    label_lower = str(row["label"]).lower()
    for d in DATA_ROOT.iterdir():
        if d.is_dir() and d.name.lower() == label_lower:
            q = d / str(row["filename"])
            if q.exists():
                return q

    raise FileNotFoundError(f"Cannot resolve raw audio for {row['filename']}")

base_df["audio_path"] = [str(resolve_audio_path(r)) for _, r in base_df.iterrows()]

print("Files:", len(base_df))
display(base_df.groupby(["label","split"]).size().unstack(fill_value=0))
print("All audio resolved:", base_df["audio_path"].map(lambda x: Path(x).exists()).all())

assert len(base_df) == 2495
assert set(base_df["split"]) == {"train","valid","test"}
assert NUM_CLASSES == 5


Files: 2495


split,test,train,valid
label,,,
cailuong,75,350,75
catru,75,347,74
chauvan,75,349,75
cheo,75,350,75
hatxam,75,350,75


All audio resolved: True


In [4]:
def load_audio_30s(path):
    y, sr = sf.read(path, dtype="float32", always_2d=False)

    if y.ndim == 2:
        y = y.mean(axis=1)

    if sr != SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=SR)

    if len(y) >= TARGET_SAMPLES:
        y = y[:TARGET_SAMPLES]
    else:
        y = np.pad(y, (0, TARGET_SAMPLES-len(y)))

    return y.astype(np.float32, copy=False)


def extract_logmel(y):
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        fmin=FMIN,
        fmax=FMAX,
        power=2.0
    )

    db = librosa.power_to_db(
        mel,
        ref=np.max,
        top_db=TOP_DB
    )

    
    norm = np.clip((db + TOP_DB) / TOP_DB, 0.0, 1.0)
    return norm.astype(np.float32)


records, errors = [], []

for _, row in tqdm(base_df.iterrows(), total=len(base_df), desc="Log-Mel + resize cache"):
    split, label, filename = row["split"], row["label"], row["filename"]
    stem = Path(filename).stem

    mel_dir = MEL_DIR / split / label
    cache_dir = CACHE_DIR / split / label
    mel_dir.mkdir(parents=True, exist_ok=True)
    cache_dir.mkdir(parents=True, exist_ok=True)

    mel_path = mel_dir / f"{stem}.npy"
    cache_path = cache_dir / f"{stem}.npy"

    try:
        if mel_path.exists():
            mel = np.load(mel_path)
        else:
            y = load_audio_30s(row["audio_path"])
            mel = extract_logmel(y)
            np.save(mel_path, mel.astype(np.float32))

        cache_ok = False
        if cache_path.exists():
            c = np.load(cache_path, mmap_mode="r")
            cache_ok = c.shape == (INPUT_H, INPUT_W) and c.dtype == np.float16

        if not cache_ok:
            x = torch.from_numpy(mel.astype(np.float32))[None,None]
            x = F.interpolate(
                x, size=(INPUT_H, INPUT_W),
                mode="bilinear", align_corners=False
            )
            resized = x[0,0].numpy().astype(np.float16)
            np.save(cache_path, resized)

        records.append({
            "split": split,
            "label": label,
            "filename": filename,
            "audio_path": row["audio_path"],
            "mel_path": str(mel_path),
            "cache_path": str(cache_path)
        })

    except Exception as e:
        errors.append({"filename": filename, "error": repr(e)})

mel_df = pd.DataFrame(records)
errors_df = pd.DataFrame(errors)

mel_df.to_csv(RUN_ROOT/"mel_manifest.csv", index=False, encoding="utf-8-sig")
errors_df.to_csv(RUN_ROOT/"mel_errors.csv", index=False, encoding="utf-8-sig")

print("OK:", len(mel_df))
print("Errors:", len(errors_df))
assert len(errors_df) == 0
assert len(mel_df) == len(base_df)

sample_mel = np.load(mel_df.iloc[0]["mel_path"])
sample_cache = np.load(mel_df.iloc[0]["cache_path"])
print("Original Mel shape:", sample_mel.shape)
print("CNN cache shape:", sample_cache.shape, sample_cache.dtype)


Log-Mel + resize cache: 100%|██████████| 2495/2495 [04:38<00:00,  8.95it/s]

OK: 2495
Errors: 0
Original Mel shape: (128, 1292)
CNN cache shape: (256, 320) float16


In [5]:
class CachedMelDataset(Dataset):
    def __init__(self, dataframe, class_to_idx, train=False):
        self.df = dataframe.reset_index(drop=True)
        self.class_to_idx = class_to_idx
        self.train = train

    def __len__(self):
        return len(self.df)

    def _augment(self, x):
        if random.random() < 0.50:
            max_shift = max(1, x.shape[-1] // 20)
            x = torch.roll(x, random.randint(-max_shift,max_shift), dims=-1)

        if random.random() < 0.50:
            width = random.randint(1, max(1,x.shape[-2]//12))
            start = random.randint(0, x.shape[-2]-width)
            x[:, start:start+width, :] = 0

        if random.random() < 0.50:
            width = random.randint(1, max(1,x.shape[-1]//10))
            start = random.randint(0, x.shape[-1]-width)
            x[:, :, start:start+width] = 0

        if random.random() < 0.40:
            x = torch.clamp(x * random.uniform(0.90,1.10), 0, 1)

        return x

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        arr = np.load(row["cache_path"]).astype(np.float32, copy=False)
        x = torch.from_numpy(arr).unsqueeze(0)
        if self.train:
            x = self._augment(x)
        return x, self.class_to_idx[row["label"]], idx


In [6]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch,out_ch,3,padding=1,bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch,out_ch,3,padding=1,bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        ]
        if dropout > 0:
            layers.append(nn.Dropout2d(dropout))
        self.block = nn.Sequential(*layers)

    def forward(self,x):
        return self.block(x)

class CNN2D_MEL(nn.Module):
    def __init__(self,n_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(1,32,0.05),
            ConvBlock(32,64,0.10),
            ConvBlock(64,128,0.15),
            ConvBlock(128,192,0.20)
        )
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(192,128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.40),
            nn.Linear(128,n_classes)
        )

    def forward(self,x):
        return self.classifier(self.pool(self.features(x)))


In [7]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

def worker_init_fn(worker_id):
    s = torch.initial_seed() % (2**32)
    np.random.seed(s)
    random.seed(s)

def metrics(y_true,y_pred):
    p,r,f,_ = precision_recall_fscore_support(
        y_true,y_pred,average="macro",zero_division=0
    )
    return {
        "accuracy": float(accuracy_score(y_true,y_pred)),
        "macro_precision": float(p),
        "macro_recall": float(r),
        "macro_f1": float(f)
    }

def make_loaders(seed):
    tr = mel_df[mel_df.split=="train"].copy()
    va = mel_df[mel_df.split=="valid"].copy()
    te = mel_df[mel_df.split=="test"].copy()

    gen = torch.Generator()
    gen.manual_seed(seed)

    common = dict(
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=NUM_WORKERS>0,
        worker_init_fn=worker_init_fn
    )
    if NUM_WORKERS:
        common["prefetch_factor"] = PREFETCH_FACTOR

    return (
        tr,va,te,
        DataLoader(CachedMelDataset(tr,class_to_idx,True), shuffle=True, generator=gen, **common),
        DataLoader(CachedMelDataset(va,class_to_idx,False), shuffle=False, **common),
        DataLoader(CachedMelDataset(te,class_to_idx,False), shuffle=False, **common)
    )


In [8]:
def run_epoch(model, loader, criterion, scaler, optimizer=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    yt, yp = [], []

    for x,y,_ in loader:
        x = x.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        y = y.to(DEVICE, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(x)
                loss = criterion(logits,y)

            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item()*x.size(0)
        yt.extend(y.detach().cpu().tolist())
        yp.extend(logits.argmax(1).detach().cpu().tolist())

    out = metrics(yt,yp)
    out["loss"] = total_loss/len(loader.dataset)
    return out

@torch.inference_mode()
def predict(model,loader):
    model.eval()
    yt,yp,probs,ids = [],[],[],[]
    for x,y,idx in loader:
        x = x.to(DEVICE,non_blocking=True,memory_format=torch.channels_last)
        with torch.autocast("cuda",dtype=torch.float16):
            logits=model(x)
        pr=torch.softmax(logits.float(),1)
        pred=pr.argmax(1)
        yt.extend(y.tolist()); yp.extend(pred.cpu().tolist())
        probs.append(pr.cpu().numpy()); ids.extend(idx.tolist())
    return np.array(yt),np.array(yp),np.concatenate(probs),np.array(ids)


In [9]:
def train_one_seed(seed):
    print("\n"+"="*76)
    print("MEL SEED",seed)
    print("="*76)

    seed_everything(seed)
    tr,va,te,train_loader,valid_loader,test_loader = make_loaders(seed)

    rdir=RESULT_ROOT/f"seed_{seed}"
    cdir=CHECKPOINT_ROOT/f"seed_{seed}"
    fdir=FIGURE_ROOT/f"seed_{seed}"
    for p in [rdir,cdir,fdir]: p.mkdir(parents=True,exist_ok=True)

    model=CNN2D_MEL(NUM_CLASSES).to(DEVICE,memory_format=torch.channels_last)
    n_params=sum(p.numel() for p in model.parameters())

    criterion=nn.CrossEntropyLoss()
    optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,mode="max",factor=0.5,patience=3,min_lr=1e-6
    )
    scaler=torch.amp.GradScaler("cuda")

    best_f1=-1.0; best_epoch=0; stale=0; history=[]
    ckpt=cdir/"best_cnn2d_mel.pt"
    start=time.time()

    for epoch in range(1,EPOCHS+1):
        tm=run_epoch(model,train_loader,criterion,scaler,optimizer)
        vm=run_epoch(model,valid_loader,criterion,scaler,None)
        scheduler.step(vm["macro_f1"])
        lr=optimizer.param_groups[0]["lr"]

        history.append({
            "epoch":epoch,"lr":lr,
            **{f"train_{k}":v for k,v in tm.items()},
            **{f"valid_{k}":v for k,v in vm.items()}
        })

        print(
            f"Seed {seed} | Epoch {epoch:02d}/{EPOCHS} | "
            f"train acc={tm['accuracy']:.4f} F1={tm['macro_f1']:.4f} | "
            f"valid acc={vm['accuracy']:.4f} F1={vm['macro_f1']:.4f} | lr={lr:.2e}"
        )

        if vm["macro_f1"] > best_f1 + MIN_DELTA:
            best_f1=vm["macro_f1"]; best_epoch=epoch; stale=0
            torch.save({
                "seed":seed,"epoch":epoch,
                "model_state_dict":model.state_dict(),
                "valid_metrics":vm,"classes":classes,
                "representation":"log_mel",
                "mel_config":{
                    "sr":SR,"n_fft":N_FFT,"hop_length":HOP_LENGTH,
                    "n_mels":N_MELS,"fmin":FMIN,"fmax":FMAX,"top_db":TOP_DB
                }
            },ckpt)
        else:
            stale+=1

        if stale>=PATIENCE:
            print("Early stopping. Best epoch:",best_epoch)
            break

    elapsed=(time.time()-start)/60
    hist=pd.DataFrame(history)
    hist.to_csv(rdir/"training_history.csv",index=False)

    eval_model=CNN2D_MEL(NUM_CLASSES).to(DEVICE,memory_format=torch.channels_last)
    state=torch.load(ckpt,map_location=DEVICE)
    eval_model.load_state_dict(state["model_state_dict"])

    ytrue,ypred,yprob,idxs=predict(eval_model,test_loader)
    testm=metrics(ytrue,ypred)

    rep=pd.DataFrame(classification_report(
        ytrue,ypred,target_names=classes,output_dict=True,zero_division=0
    )).T
    rep.to_csv(rdir/"classification_report.csv",encoding="utf-8-sig")

    cm=pd.DataFrame(confusion_matrix(ytrue,ypred),index=classes,columns=classes)
    cm.to_csv(rdir/"confusion_matrix.csv",encoding="utf-8-sig")

    rows=[]
    for n,idx in enumerate(idxs):
        row=te.iloc[int(idx)]
        d={
            "filename":row.filename,
            "label":row.label,
            "predicted_label":idx_to_class[int(ypred[n])],
            "correct":bool(ytrue[n]==ypred[n]),
            "confidence":float(yprob[n].max())
        }
        for cid,cname in idx_to_class.items():
            d[f"prob_{cname}"]=float(yprob[n,cid])
        rows.append(d)
    pd.DataFrame(rows).to_csv(rdir/"test_predictions.csv",index=False,encoding="utf-8-sig")

    plt.figure(figsize=(9,5))
    plt.plot(hist.epoch,hist.train_macro_f1,label="train macro-F1")
    plt.plot(hist.epoch,hist.valid_macro_f1,label="valid macro-F1")
    plt.xlabel("Epoch"); plt.ylabel("Macro-F1"); plt.title(f"Log-Mel — Seed {seed}")
    plt.legend(); plt.tight_layout()
    plt.savefig(fdir/"macro_f1_curve.png",dpi=160); plt.close()

    summary={
        "seed":seed,
        "best_epoch":best_epoch,
        "best_valid_macro_f1":best_f1,
        "test_accuracy":testm["accuracy"],
        "test_macro_precision":testm["macro_precision"],
        "test_macro_recall":testm["macro_recall"],
        "test_macro_f1":testm["macro_f1"],
        "training_minutes":elapsed,
        "parameters":n_params
    }
    with open(rdir/"summary.json","w",encoding="utf-8") as f:
        json.dump(summary,f,indent=2,ensure_ascii=False)

    print(f"TEST: Acc={testm['accuracy']:.4f} | Macro-F1={testm['macro_f1']:.4f} | {elapsed:.2f} min")

    del model,eval_model,train_loader,valid_loader,test_loader
    gc.collect(); torch.cuda.empty_cache()
    return summary,rep


In [10]:
summaries=[]
reports={}
overall=time.time()

for seed in SEEDS:
    s,r=train_one_seed(seed)
    summaries.append(s)
    reports[seed]=r

total_minutes=(time.time()-overall)/60
print(f"\nAll 3 Mel seeds completed in {total_minutes:.2f} minutes.")



MEL SEED 42
Seed 42 | Epoch 01/50 | train acc=0.4341 F1=0.4036 | valid acc=0.2005 F1=0.0668 | lr=3.00e-04
Seed 42 | Epoch 02/50 | train acc=0.5968 F1=0.5623 | valid acc=0.2005 F1=0.0670 | lr=3.00e-04
Seed 42 | Epoch 03/50 | train acc=0.6684 F1=0.6493 | valid acc=0.4759 F1=0.4541 | lr=3.00e-04
Seed 42 | Epoch 04/50 | train acc=0.7062 F1=0.6954 | valid acc=0.7647 F1=0.7529 | lr=3.00e-04
Seed 42 | Epoch 05/50 | train acc=0.7142 F1=0.7017 | valid acc=0.7193 F1=0.6976 | lr=3.00e-04
Seed 42 | Epoch 06/50 | train acc=0.7297 F1=0.7251 | valid acc=0.7674 F1=0.7688 | lr=3.00e-04
Seed 42 | Epoch 07/50 | train acc=0.7480 F1=0.7436 | valid acc=0.7754 F1=0.7642 | lr=3.00e-04
Seed 42 | Epoch 08/50 | train acc=0.7698 F1=0.7674 | valid acc=0.8610 F1=0.8595 | lr=3.00e-04
Seed 42 | Epoch 09/50 | train acc=0.7978 F1=0.7948 | valid acc=0.8663 F1=0.8640 | lr=3.00e-04
Seed 42 | Epoch 10/50 | train acc=0.7910 F1=0.7878 | valid acc=0.8636 F1=0.8636 | lr=3.00e-04
Seed 42 | Epoch 11/50 | train acc=0.8018 F1=0.7

In [11]:
summary_df=pd.DataFrame(summaries)
summary_df.to_csv(RESULT_ROOT/"multiseed_per_seed.csv",index=False,encoding="utf-8-sig")
display(summary_df)

rows=[]
for m in ["test_accuracy","test_macro_precision","test_macro_recall","test_macro_f1"]:
    v=summary_df[m].to_numpy(float)
    rows.append({"metric":m,"mean":v.mean(),"std":v.std(ddof=1),"min":v.min(),"max":v.max()})

aggregate_df=pd.DataFrame(rows)
aggregate_df.to_csv(RESULT_ROOT/"multiseed_mean_std.csv",index=False,encoding="utf-8-sig")
display(aggregate_df)


,seed,best_epoch,best_valid_macro_f1,test_accuracy,test_macro_precision,test_macro_recall,test_macro_f1,training_minutes,parameters
0,42,43,0.975842,0.973333,0.974224,0.973333,0.973011,2.285690,865957
1,123,37,0.983892,0.968000,0.969097,0.968000,0.967592,1.956214,865957
2,2026,41,0.978535,0.973333,0.973620,0.973333,0.973047,2.113210,865957


,metric,mean,std,min,max
0,test_accuracy,0.971556,0.003079,0.968000,0.973333
1,test_macro_precision,0.972313,0.002802,0.969097,0.974224
2,test_macro_recall,0.971556,0.003079,0.968000,0.973333
3,test_macro_f1,0.971217,0.003139,0.967592,0.973047


In [12]:
rows=[]
for cname in classes:
    for metric in ["precision","recall","f1-score"]:
        v=np.array([reports[s].loc[cname,metric] for s in SEEDS],dtype=float)
        rows.append({
            "class":cname,"metric":metric,
            "mean":v.mean(),"std":v.std(ddof=1),
            "min":v.min(),"max":v.max()
        })

per_class_df=pd.DataFrame(rows)
per_class_df.to_csv(RESULT_ROOT/"multiseed_per_class.csv",index=False,encoding="utf-8-sig")
display(per_class_df)


,class,metric,mean,std,min,max
0,cailuong,precision,0.995169,8.367395e-03,0.985507,1.000000
1,cailuong,recall,0.893333,1.333333e-02,0.880000,0.906667
2,cailuong,f1-score,0.941426,4.568034e-03,0.936170,0.944444
3,catru,precision,0.986842,0.000000e+00,0.986842,0.986842
4,catru,recall,1.000000,0.000000e+00,1.000000,1.000000
5,catru,f1-score,0.993377,1.359740e-16,0.993377,0.993377
6,chauvan,precision,0.961538,0.000000e+00,0.961538,0.961538
7,chauvan,recall,1.000000,0.000000e+00,1.000000,1.000000
8,chauvan,f1-score,0.980392,0.000000e+00,0.980392,0.980392
9,cheo,precision,0.982570,7.399397e-03,0.974026,0.986842


In [13]:
acc=summary_df.test_accuracy.to_numpy(float)
f1=summary_df.test_macro_f1.to_numpy(float)

final={
    "experiment":"CNN2D_LogMel_30s_multiseed_v1",
    "seeds":SEEDS,
    "fixed_split":True,
    "representation":"log_mel",
    "mel_config":{
        "sr":SR,"n_fft":N_FFT,"hop_length":HOP_LENGTH,
        "n_mels":N_MELS,"fmin":FMIN,"fmax":FMAX,
        "power":2.0,"top_db":TOP_DB
    },
    "cnn_input_shape":[1,INPUT_H,INPUT_W],
    "batch_size":BATCH_SIZE,
    "num_workers":NUM_WORKERS,
    "amp_fp16":True,
    "channels_last":True,
    "resize_cache":True,
    "test_accuracy_mean":float(acc.mean()),
    "test_accuracy_std":float(acc.std(ddof=1)),
    "test_macro_f1_mean":float(f1.mean()),
    "test_macro_f1_std":float(f1.std(ddof=1)),
    "total_training_minutes":float(total_minutes)
}

with open(RESULT_ROOT/"multiseed_summary.json","w",encoding="utf-8") as f:
    json.dump(final,f,indent=2,ensure_ascii=False)

comparison=pd.DataFrame([
    {
        "representation":"STFT",
        "accuracy_mean":0.9724444444444446,
        "accuracy_std":0.013684270505191663,
        "macro_f1_mean":0.9722214759351866,
        "macro_f1_std":0.013962992701270922
    },
    {
        "representation":"Log-Mel",
        "accuracy_mean":acc.mean(),
        "accuracy_std":acc.std(ddof=1),
        "macro_f1_mean":f1.mean(),
        "macro_f1_std":f1.std(ddof=1)
    }
])
comparison.to_csv(RESULT_ROOT/"STFT_vs_MEL.csv",index=False,encoding="utf-8-sig")
display(comparison)

print(json.dumps(final,indent=2,ensure_ascii=False))


,representation,accuracy_mean,accuracy_std,macro_f1_mean,macro_f1_std
0,STFT,0.972444,0.013684,0.972221,0.013963
1,Log-Mel,0.971556,0.003079,0.971217,0.003139


{
  "experiment": "CNN2D_LogMel_30s_multiseed_v1",
  "seeds": [
    42,
    123,
    2026
  ],
  "fixed_split": true,
  "representation": "log_mel",
  "mel_config": {
    "sr": 22050,
    "n_fft": 2048,
    "hop_length": 512,
    "n_mels": 128,
    "fmin": 0,
    "fmax": 11025.0,
    "power": 2.0,
    "top_db": 80
  },
  "cnn_input_shape": [
    1,
    256,
    320
  ],
  "batch_size": 96,
  "num_workers": 8,
  "amp_fp16": true,
  "channels_last": true,
  "resize_cache": true,
  "test_accuracy_mean": 0.9715555555555556,
  "test_accuracy_std": 0.00307920143567805,
  "test_macro_f1_mean": 0.9712166561879062,
  "test_macro_f1_std": 0.0031393660677680644,
  "total_training_minutes": 6.4304152727127075
}


In [14]:
print("="*72)
print("LOG-MEL 30s + CNN2D — FINAL")
print("="*72)
for r in summaries:
    print(
        f"Seed {r['seed']:4d} | Acc={r['test_accuracy']:.4f} | "
        f"Macro-F1={r['test_macro_f1']:.4f} | "
        f"Best epoch={r['best_epoch']:02d} | {r['training_minutes']:.2f} min"
    )

print(f"\nMEL TEST ACCURACY = {acc.mean():.4f} ± {acc.std(ddof=1):.4f}")
print(f"MEL TEST MACRO-F1 = {f1.mean():.4f} ± {f1.std(ddof=1):.4f}")
print("\nResults:", RESULT_ROOT)


LOG-MEL 30s + CNN2D — FINAL
Seed   42 | Acc=0.9733 | Macro-F1=0.9730 | Best epoch=43 | 2.29 min
Seed  123 | Acc=0.9680 | Macro-F1=0.9676 | Best epoch=37 | 1.96 min
Seed 2026 | Acc=0.9733 | Macro-F1=0.9730 | Best epoch=41 | 2.11 min

MEL TEST ACCURACY = 0.9716 ± 0.0031
MEL TEST MACRO-F1 = 0.9712 ± 0.0031

Results: /mnt/d/Data/VNTM3/_cnn2d_mel30_multiseed_v1/results
